
# Held-out spatial-minus-temporal top-\(m\) heatmaps

This notebook recreates the logic/layout of the submitted spatial-minus-temporal heatmap using **participant-held-out top-\(m\) results for both analysis orientations**.

Important differences from the submitted figure:

- archetypes are selected on one participant half and evaluated on the other;
- spatial and temporal top-\(m\) subsets are selected independently within their own decompositions;
- A→B and B→A directions are averaged within each outer split before the model-order mean is computed;
- spatial and temporal models are compared only at **exactly matched normalized component ratios**, where \(K_{spatial}/700 = K_{temporal}/300\);
- saturated cells with \(m>K\) are masked;
- no interpolation is used.


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SPATIAL_RUN_CSV = Path(
    "msaa_heldout_topm_outputs_spatial_across_fullgrid/"
    "heldout_topm_crossfit_runs.csv"
)
TEMPORAL_RUN_CSV = Path(
    "msaa_heldout_topm_outputs_temporal_across_fullgrid/"
    "heldout_topm_crossfit_runs.csv"
)

OUTPUT_DIR = Path("outputs/paper_supplement")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PNG = OUTPUT_DIR / "Supplement_heldout_spatial_minus_temporal_heatmaps.png"
FIG_PDF = OUTPUT_DIR / "Supplement_heldout_spatial_minus_temporal_heatmaps.pdf"
SOURCE_CSV = OUTPUT_DIR / "Supplement_heldout_spatial_minus_temporal_heatmaps_source.csv"

CONDITIONS = ["intact", "word", "rest"]
CONDITION_TITLES = {
    "intact": "Intact",
    "word": "Word-scrambled",
    "rest": "Rest",
}
TOP_M_VALUES = [1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 40, 50, 75, 100]

DIMENSIONS = {"spatial": 700, "temporal": 300}

# Optional cell annotations on the intact panel.
ANNOTATE_INTACT_N = 10


In [ ]:

if not SPATIAL_RUN_CSV.exists():
    raise FileNotFoundError(
        f"Missing {SPATIAL_RUN_CSV}\n"
        "Run 004b_expand_spatial_heldout_topm_fullgrid.ipynb first."
    )
if not TEMPORAL_RUN_CSV.exists():
    raise FileNotFoundError(
        f"Missing {TEMPORAL_RUN_CSV}\n"
        "Run 004_temporal_heldout_topm_fullgrid.ipynb first."
    )

spatial_runs = pd.read_csv(SPATIAL_RUN_CSV)
temporal_runs = pd.read_csv(TEMPORAL_RUN_CSV)

print("Spatial rows:", len(spatial_runs))
print("Temporal rows:", len(temporal_runs))
print("Spatial top-m:", sorted(spatial_runs["top_m"].astype(int).unique()))
print("Temporal top-m:", sorted(temporal_runs["top_m"].astype(int).unique()))


In [ ]:

def sem(x):
    x = pd.Series(x).dropna()
    return np.nan if len(x) <= 1 else float(x.std(ddof=1) / np.sqrt(len(x)))

def summarize_outer_paired(run_df, analysis):
    df = run_df.copy()
    df = df[df["analysis_type"].astype(str) == analysis].copy()

    # Average A->B and B->A within each outer participant split first.
    outer = (
        df.groupby(
            ["condition", "K", "top_m", "outer_rep"],
            as_index=False,
        )
        .agg(
            accuracy=("mean", "mean"),
            full_accuracy=("full_mean_same_eval", "mean"),
        )
    )

    return (
        outer.groupby(["condition", "K", "top_m"], as_index=False)
        .agg(
            accuracy=("accuracy", "mean"),
            sem=("accuracy", sem),
            n_outer=("accuracy", "size"),
            full_accuracy=("full_accuracy", "mean"),
        )
    )

spatial = summarize_outer_paired(spatial_runs, "spatial")
temporal = summarize_outer_paired(temporal_runs, "temporal")

display(spatial.head())
display(temporal.head())


In [ ]:

# Exact spatial/temporal normalized component-ratio matches.
# Every row satisfies spatial_K / 700 == temporal_K / 300 exactly.
EXACT_PAIRS = [
    (7,   3),    # 0.01
    (14,  6),    # 0.02
    (21,  9),    # 0.03
    (35, 15),    # 0.05
    (42, 18),    # 0.06
    (63, 27),    # 0.09
    (70, 30),    # 0.10
    (105, 45),   # 0.15
    (126, 54),   # 0.18
    (140, 60),   # 0.20
    (175, 75),   # 0.25
    (189, 81),   # 0.27
    (210, 90),   # 0.30
    (252, 108),  # 0.36
    (700, 300),  # 1.00
]

pair_table = pd.DataFrame(EXACT_PAIRS, columns=["spatial_K", "temporal_K"])
pair_table["spatial_ratio"] = (
    pair_table["spatial_K"] / DIMENSIONS["spatial"]
)
pair_table["temporal_ratio"] = (
    pair_table["temporal_K"] / DIMENSIONS["temporal"]
)
pair_table["matched_ratio"] = pair_table["spatial_ratio"]
pair_table["ratio_difference"] = np.abs(
    pair_table["spatial_ratio"] - pair_table["temporal_ratio"]
)

# These should be mathematically identical; fail loudly if the table is edited
# incorrectly in the future.
if not np.allclose(
    pair_table["spatial_ratio"],
    pair_table["temporal_ratio"],
    rtol=0.0,
    atol=1e-12,
):
    raise ValueError("EXACT_PAIRS contains a non-exact spatial/temporal ratio match.")

# Report any model orders that are not yet present in the held-out result files.
spatial_available = set(spatial["K"].astype(int))
temporal_available = set(temporal["K"].astype(int))
missing_spatial = sorted(set(pair_table["spatial_K"]) - spatial_available)
missing_temporal = sorted(set(pair_table["temporal_K"]) - temporal_available)

if missing_spatial:
    print("WARNING: missing spatial held-out K values:", missing_spatial)
if missing_temporal:
    print("WARNING: missing temporal held-out K values:", missing_temporal)

display(pair_table)


In [ ]:

def build_difference_table(spatial, temporal, pair_table):
    rows = []

    for p in pair_table.itertuples(index=False):
        sK = int(p.spatial_K)
        tK = int(p.temporal_K)

        for condition in CONDITIONS:
            for top_m in TOP_M_VALUES:
                # Match the submitted heatmap's saturation mask.
                if top_m > sK or top_m > tK:
                    continue

                s = spatial[
                    (spatial["condition"].astype(str) == condition)
                    & (spatial["K"].astype(int) == sK)
                    & (spatial["top_m"].astype(int) == top_m)
                ]
                t = temporal[
                    (temporal["condition"].astype(str) == condition)
                    & (temporal["K"].astype(int) == tK)
                    & (temporal["top_m"].astype(int) == top_m)
                ]

                if len(s) == 0 or len(t) == 0:
                    continue

                s_acc = float(s["accuracy"].iloc[0])
                t_acc = float(t["accuracy"].iloc[0])

                rows.append({
                    "condition": condition,
                    "top_m": int(top_m),
                    "spatial_K": sK,
                    "temporal_K": tK,
                    "spatial_ratio": float(p.spatial_ratio),
                    "temporal_ratio": float(p.temporal_ratio),
                    "matched_ratio": float(p.matched_ratio),
                    "ratio_difference": float(p.ratio_difference),
                    "spatial_heldout_accuracy": s_acc,
                    "temporal_heldout_accuracy": t_acc,
                    "difference_spatial_minus_temporal": s_acc - t_acc,
                    "spatial_sem": float(s["sem"].iloc[0]),
                    "temporal_sem": float(t["sem"].iloc[0]),
                    "spatial_n_outer": int(s["n_outer"].iloc[0]),
                    "temporal_n_outer": int(t["n_outer"].iloc[0]),
                })

    return pd.DataFrame(rows)

diff_df = build_difference_table(spatial, temporal, pair_table)
diff_df.to_csv(SOURCE_CSV, index=False)

print("Difference rows:", len(diff_df))
print("Saved:", SOURCE_CSV)

coverage = (
    diff_df.groupby("condition")
    .agg(
        n_cells=("difference_spatial_minus_temporal", "size"),
        n_pairs=("spatial_K", "nunique"),
        n_top_m=("top_m", "nunique"),
    )
)
display(coverage)


In [ ]:

def make_heat(condition):
    heat = np.full((len(TOP_M_VALUES), len(pair_table)), np.nan)

    sub = diff_df[diff_df["condition"] == condition]

    for i, m in enumerate(TOP_M_VALUES):
        for j, p in enumerate(pair_table.itertuples(index=False)):
            vals = sub[
                (sub["top_m"].astype(int) == int(m))
                & (sub["spatial_K"].astype(int) == int(p.spatial_K))
                & (sub["temporal_K"].astype(int) == int(p.temporal_K))
            ]["difference_spatial_minus_temporal"]
            if len(vals):
                heat[i, j] = float(vals.iloc[0])

    return heat

heats = {condition: make_heat(condition) for condition in CONDITIONS}
finite = np.concatenate([
    h[np.isfinite(h)] for h in heats.values() if np.isfinite(h).any()
])
if len(finite) == 0:
    raise RuntimeError("No matched held-out cells were available to plot.")

max_abs = float(np.max(np.abs(finite)))
print("Shared color range:", -max_abs, max_abs)


In [ ]:

fig, axes = plt.subplots(
    3, 1,
    figsize=(12, 9),
    sharex=True,
    constrained_layout=False,
)

ims = []
for ax, condition in zip(axes, CONDITIONS):
    heat = heats[condition]

    im = ax.imshow(
        heat,
        aspect="auto",
        origin="lower",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-max_abs,
        vmax=max_abs,
    )
    ims.append(im)

    ax.set_title(CONDITION_TITLES[condition], fontsize=15, fontweight="bold")
    ax.set_ylabel("top-m")
    ax.set_yticks(np.arange(len(TOP_M_VALUES)))
    ax.set_yticklabels(TOP_M_VALUES)

# Exact matched-ratio x-axis labels.
labels = [f"{x:.2f}" for x in pair_table["matched_ratio"]]
axes[-1].set_xticks(np.arange(len(pair_table)))
axes[-1].set_xticklabels(labels, rotation=45, ha="right")
axes[-1].set_xlabel("Matched component ratio (K / D)")

# Annotate the largest intact differences, similar to the submitted figure.
intact_heat = heats["intact"]
valid = np.argwhere(np.isfinite(intact_heat))
if len(valid):
    vals = np.array([abs(intact_heat[i, j]) for i, j in valid])
    order = np.argsort(vals)[-ANNOTATE_INTACT_N:]
    for idx in order:
        i, j = valid[idx]
        val = intact_heat[i, j]
        sK = int(pair_table.iloc[j]["spatial_K"])
        tK = int(pair_table.iloc[j]["temporal_K"])
        axes[0].text(
            j, i, f"S{sK}/T{tK}",
            ha="center", va="center",
            fontsize=7,
            fontweight="bold",
            color="white" if abs(val) > 0.55 * max_abs else "black",
        )

fig.suptitle(
    "Held-out spatial minus temporal top-m timepoint identification",
    fontsize=17,
    fontweight="bold",
    y=0.985,
)

fig.subplots_adjust(
    left=0.10,
    right=0.88,
    top=0.92,
    bottom=0.10,
    hspace=0.18,
)

cax = fig.add_axes([0.90, 0.20, 0.018, 0.60])
cb = fig.colorbar(ims[0], cax=cax)
cb.set_label("Spatial - temporal held-out accuracy")

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)



## Interpretation

This figure should be interpreted more cautiously than the submitted same-data heatmap.

A positive cell means that, at that matched spatial/temporal representational ratio and subset size, the **participant-held-out spatial top-\(m\)** reconstruction had higher mean timepoint-identification accuracy than the **participant-held-out temporal top-\(m\)** reconstruction.

Because spatial and temporal archetypes are ranked independently within their respective decompositions, the comparison is between the best reproducibly selected sparse subsets available to each orientation, not between identically indexed components.

The main question is whether the broad spatial-versus-temporal asymmetry from the submitted figure survives when selection and evaluation are separated across participants.
